In [2]:
import sys
sys.path.insert(0, '../config')

# Limpa cache se existir
if 'config' in sys.modules:
    del sys.modules['config']


from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from config import S3_BRONZE, S3_BUCKET, SPARK_APP_NAME, SPARK_MASTER

In [3]:
#spark = get_spark()

spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

26/09/02 16:58:48 WARN Utils: Your hostname, hugo resolves to a loopback address: 127.0.1.1; using 10.159.141.203 instead (on interface wlp2s0)
26/09/02 16:58:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hugo/Desktop/Project/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hugo/.ivy2/cache
The jars for the packages stored in: /home/hugo/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-596e797e-7d2e-496a-a595-e1482bb08efc;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.261 in central
:: resolution report :: resolve 453ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.261 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 by [com.amazonaws#aws-java-sdk-bundle;1.12.261] in [default]
	------------------------------------------------------------------

# Lendo bases do S3

In [4]:
# Lê dados do S3
df_product_item = spark.read.parquet(f"{S3_BRONZE}/product_item", header=True, inferSchema=True)

df_purchase = spark.read.parquet(f"{S3_BRONZE}/purchase", header=True, inferSchema=True)


26/09/02 16:58:56 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


# Documentação

df_product_item
  - prod_item_id:           id do item do produtor
  - product_id:             id do produto
  - item_quantity:          Quantidade de items
  - purchase_value:         Valor da compra
  - prod_item_partition:    partition_item m traga aqui mesmo

df_purchase:
  - purchase_id:      ID da Compra
  - buyer_id:         ID do Comprador
  - prod_item_id:     Id  
  - release_date:     Data da liberação do produto para o comprador


# Creating View para usar em SQL

In [5]:

df_purchase.createOrReplaceTempView("df_purchase")

df_product_item.createOrReplaceTempView("df_product_item")

# Show nas tabelas

In [6]:
df_purchase.show()

+-----------+--------+------------+----------+------------+-----------+------------------+-------------------+--------------------+---------------+--------------------+----------------+
|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|purchase_partition|prod_item_partition|purchase_total_value|purchase_status|transaction_datetime|transaction_date|
+-----------+--------+------------+----------+------------+-----------+------------------+-------------------+--------------------+---------------+--------------------+----------------+
|       1000|  719570|          48|2021-01-13|  2021-01-14|     100004|                 3|                  1|              2678.7|    REEMBOLSADA| 2021-01-14 02:37:27|      2021-01-14|
|       1001|  760800|          36|2021-02-17|  2021-02-21|     100010|                 1|                  3|             2109.95|    REEMBOLSADA| 2021-02-21 13:14:28|      2021-02-21|
|       1002|  173032|          22|2021-01-04|  2021-01-05|     100012

In [7]:
df_product_item.show()

+------------+-------------------+----------+-------------+--------------+--------------------+----------------+
|prod_item_id|prod_item_partition|product_id|item_quantity|purchase_value|transaction_datetime|transaction_date|
+------------+-------------------+----------+-------------+--------------+--------------------+----------------+
|           1|                  0|        78|          179|        381.58| 2022-08-05 11:42:44|      2022-08-05|
|           2|                  1|        89|          154|        249.64| 2020-02-09 16:27:36|      2020-02-09|
|           3|                  5|        99|          141|        443.37| 2022-06-07 12:11:03|      2022-06-07|
|           4|                  1|        67|           97|         77.58| 2020-11-24 16:53:31|      2020-11-24|
|           5|                  2|        84|          153|        489.12| 2021-07-10 18:11:24|      2021-07-10|
|           6|                  1|        63|          149|         31.54| 2020-05-05 05:18:17| 

# Questions

## 1 - Quais são os 50 maiores produtores em faturamento($) de 2021?
Filtros
- Ano de 2021
- Somente compra aprovada: purchase_status == APROVADA


In [ ]:
#Pq eu usei o purchase_total_value da tabela purchase esse provavelmente é o campo que é a soma de todas as taxas, subisidios, cupons que foram aplicados na compra.
# Como estamos olhando para o faturamento total do produtor podemos olhar por esse campo, mas se realmente quisermos saber o valor que o produtor vai receber no final, ai precisamos de outras tabelas na qual conseguimos identificar todas as taxas subsidios e cupons que foram aplicados nessa compra.

# Explicação da query:
#   OLhei a tabela purchase filtrando somente o ano de 2021 p.order_date between '2021-01-01' AND '2021-12-31' onde o status da compra purchase_status == 'APROVADA' que são os filtros que vi no diagrama, no fianl ordenei descrecente pelo valor do faturamento purchase_total_value e limite somente para os primeiros 50 produtores.
# Assim temos os 50 maiores produtos do ano de 2021.

query_sql = """
                SELECT
                    year(p.order_date) as year_of_purchase,
                    p.producer_id,
                    round(sum(p.purchase_total_value),2) as faturamento
                FROM df_purchase p
                WHERE p.order_date between '2021-01-01' AND '2021-12-31'
                and p.purchase_status = 'APROVADA'
                GROUP BY ALL
                ORDER BY faturamento DESC
                limit 50;
                """

spark.sql(query_sql).show()

+----------------+-----------+-----------+
|year_of_purchase|producer_id|faturamento|
+----------------+-----------+-----------+
|            2021|     100004|    9728.24|
|            2021|     100002|    8930.99|
|            2021|     100003|     8063.8|
|            2021|     100005|    7628.81|
|            2021|     100013|    7351.29|
|            2021|     100011|    6671.55|
|            2021|     100012|    6428.69|
|            2021|     100015|    6274.58|
|            2021|     100006|    6014.36|
|            2021|     100014|    5955.41|
|            2021|     100001|    5617.15|
|            2021|     100010|    5145.35|
|            2021|     100009|    3572.27|
|            2021|     100008|    1389.52|
|            2021|     100007|     680.28|
+----------------+-----------+-----------+



## 1 - Quais são os 2 produtos que mais faturaram ($) de cada produtor?

In [ ]:
"""Explicação da query:

- Na CTE faturamento_por_produto estou fazendo o join entre as duas tabelas para pegar qual foi o produtor referente a product_item e incluir o filtro na purchase das compras que foram aprovadas, como no enunciado não deixou claro o ano da analise, deixei sem a data order_date, assim pegando todo o período da base, mas se for no mesmo ano de 2021 é só incluir (p.order_date BETWEEN '2021-01-01' AND '2021-12-31')
- Na CTE ranked trouxe os acmpos producer_id, product_id, faturamento e rankiei o faturamento particionando por producer_id e ordernando decrescente o faturamento (pegando do maior para o menor) assim temos um ranqueamento do produtor e produto com base no faturamento.
- No Select Final trouxe os campos producer_id, product_id, faturamento e o rank filtrando somente os 2 maiores faturamentos dos produtos.

Assim conseguimos, trazendo os 2 produtos que mais faturaram de cada produtor.

"""
quer_sql_2 = """
                WITH faturamento_por_produto AS (
                    SELECT
                        p.producer_id,
                        pi.product_id,
                        ROUND(SUM(pi.purchase_value), 2) AS faturamento
                    FROM df_purchase p
                    JOIN df_product_item pi
                        ON pi.prod_item_id = p.prod_item_id AND pi.prod_item_partition = p.prod_item_partition
                    WHERE 1=1 
                    AND p.purchase_status = 'APROVADA'
                    GROUP BY ALL
                ),
                ranked AS (
                    SELECT
                        producer_id,
                        product_id,
                        faturamento,
                        ROW_NUMBER() OVER (
                            PARTITION BY producer_id
                            ORDER BY faturamento DESC
                        ) AS rank
                    FROM faturamento_por_produto
                )
                SELECT
                    producer_id,
                    product_id,
                    faturamento, rank
                FROM ranked
                WHERE rank <= 2
                ORDER BY producer_id, rank
            """

spark.sql(quer_sql_2).show()

+-----------+----------+-----------+----+
|producer_id|product_id|faturamento|rank|
+-----------+----------+-----------+----+
|     100001|        91|     205.78|   1|
|     100002|        72|     213.12|   1|
|     100002|        81|     160.68|   2|
|     100004|        80|     438.32|   1|
|     100007|        97|     264.05|   1|
|     100008|        81|      44.49|   1|
|     100010|        69|     403.93|   1|
|     100011|        80|     876.64|   1|
|     100011|        82|     183.05|   2|
|     100013|        74|     464.24|   1|
|     100014|        99|     443.37|   1|
|     100014|        66|     285.83|   2|
+-----------+----------+-----------+----+

